# Day 050 | Regression Metrics (R², RMSE, MAE)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| R² (Coefficient of Determination) | Fraction of variance explained | "Model explains 92% of bearing capacity variance" |
| Adjusted R² | R² penalized for number of features | Prevents overfitting with too many features |
| RMSE (Root Mean Squared Error) | Average prediction error (same units) | "Off by 15 kPa on average" |
| MAE (Mean Absolute Error) | Robust average error | When a few extreme errors should not dominate |
| MAPE (Mean Absolute Percentage Error) | Error as a percentage of actual | "Predictions are off by 8% on average" |
| Explained Variance | Variance captured by predictions | Similar to R² but unbiased for mean |

---

## Our Geotechnical Problem

Two models predict **ultimate bearing capacity (qu)** of shallow foundations.  
Which model is better? The answer depends on **which metric** you use.

- **Model A:** Random Forest  
- **Model B:** XGBoost

We will compute every major regression metric and understand when each one matters most.

---

## Part 1: Generate Synthetic Bearing Capacity Data

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS AND DATA GENERATION
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n_samples = 400

# Features for bearing capacity prediction
footing_width = np.random.uniform(0.5, 4.0, n_samples)        # B (m)
footing_depth = np.random.uniform(0.5, 3.0, n_samples)        # Df (m)
cohesion = np.random.uniform(5, 80, n_samples)                 # c (kPa)
friction_angle = np.random.uniform(15, 40, n_samples)          # phi (degrees)
unit_weight = np.random.uniform(16, 22, n_samples)             # gamma (kN/m3)
spt_n = np.random.uniform(5, 50, n_samples)                    # SPT N-value

# Bearing capacity factors (simplified Terzaghi)
phi_rad = np.radians(friction_angle)
Nq = np.exp(np.pi * np.tan(phi_rad)) * np.tan(np.radians(45 + friction_angle/2))**2
Nc = (Nq - 1) / np.tan(phi_rad)
Ngamma = 2 * (Nq + 1) * np.tan(phi_rad)

# Ultimate bearing capacity (kPa)
qu = (
    cohesion * Nc
    + unit_weight * footing_depth * Nq
    + 0.5 * unit_weight * footing_width * Ngamma
    + np.random.normal(0, 30, n_samples)
)
qu = np.clip(qu, 50, 5000)

df = pd.DataFrame({
    'B_m': footing_width,
    'Df_m': footing_depth,
    'c_kPa': cohesion,
    'phi_deg': friction_angle,
    'gamma_kNm3': unit_weight,
    'SPT_N': spt_n,
    'qu_kPa': qu
})

df.to_csv('bearing_capacity_data.csv', index=False)
print(f"Dataset shape: {df.shape}")
print(f"\nTarget (qu) statistics:")
print(df['qu_kPa'].describe().round(1))
df.head()

---

## Part 2: Train Two Competing Models

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN TWO MODELS FOR COMPARISON
# ─────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

features = ['B_m', 'Df_m', 'c_kPa', 'phi_deg', 'gamma_kNm3', 'SPT_N']
X = df[features]
y = df['qu_kPa']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model A: Random Forest
model_a = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
model_a.fit(X_train, y_train)
y_pred_a = model_a.predict(X_test)

# Model B: XGBoost
model_b = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
model_b.fit(X_train, y_train)
y_pred_b = model_b.predict(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"\nModel A: Random Forest  ({len(model_a.estimators_)} trees)")
print(f"Model B: XGBoost        ({model_b.n_estimators} boosters)")

---

## Part 3: R² and Adjusted R²

**R² (Coefficient of Determination):**

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2} = 1 - \frac{SS_{res}}{SS_{tot}}$$

- R² = 1.0 → Perfect prediction  
- R² = 0.0 → Model is as good as predicting the mean  
- R² < 0   → Model is worse than predicting the mean

**Adjusted R²** penalizes adding useless features:

$$R^2_{adj} = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

In [ ]:
# ─────────────────────────────────────────────────────
# R² AND ADJUSTED R²
# ─────────────────────────────────────────────────────

from sklearn.metrics import r2_score

def adjusted_r2(y_true, y_pred, n_features):
    """Compute adjusted R² penalized for number of features."""
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

r2_a = r2_score(y_test, y_pred_a)
r2_b = r2_score(y_test, y_pred_b)
adj_r2_a = adjusted_r2(y_test, y_pred_a, len(features))
adj_r2_b = adjusted_r2(y_test, y_pred_b, len(features))

print("R² (Coefficient of Determination):")
print(f"  Model A (RF):      R² = {r2_a:.4f}  |  Adj R² = {adj_r2_a:.4f}")
print(f"  Model B (XGBoost): R² = {r2_b:.4f}  |  Adj R² = {adj_r2_b:.4f}")
print()
print("Interpretation:")
print(f"  Model A explains {r2_a*100:.1f}% of bearing capacity variance.")
print(f"  Model B explains {r2_b*100:.1f}% of bearing capacity variance.")
print(f"  Adj R² is slightly lower — penalizes for {len(features)} features.")

---

## Part 4: RMSE — Root Mean Squared Error

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

- Same units as target (kPa)  
- Sensitive to large errors (squares them)  
- **Best when large errors are unacceptable** (structural safety)

In [ ]:
# ─────────────────────────────────────────────────────
# RMSE — Penalizes large errors heavily
# ─────────────────────────────────────────────────────

from sklearn.metrics import mean_squared_error

rmse_a = np.sqrt(mean_squared_error(y_test, y_pred_a))
rmse_b = np.sqrt(mean_squared_error(y_test, y_pred_b))

print("RMSE (Root Mean Squared Error):")
print(f"  Model A (RF):      RMSE = {rmse_a:.2f} kPa")
print(f"  Model B (XGBoost): RMSE = {rmse_b:.2f} kPa")
print()
print("Interpretation:")
print(f"  Model A is off by ~{rmse_a:.0f} kPa on average (weighted toward large errors).")
print(f"  Mean qu = {y_test.mean():.0f} kPa, so RMSE is {rmse_a/y_test.mean()*100:.1f}% of the mean.")

---

## Part 5: MAE — Mean Absolute Error

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

- Same units as target (kPa)  
- **Robust to outliers** — does not square errors  
- Better when you want the "typical" error without outlier influence

In [ ]:
# ─────────────────────────────────────────────────────
# MAE — Robust to outlier errors
# ─────────────────────────────────────────────────────

from sklearn.metrics import mean_absolute_error

mae_a = mean_absolute_error(y_test, y_pred_a)
mae_b = mean_absolute_error(y_test, y_pred_b)

print("MAE (Mean Absolute Error):")
print(f"  Model A (RF):      MAE = {mae_a:.2f} kPa")
print(f"  Model B (XGBoost): MAE = {mae_b:.2f} kPa")
print()
print("RMSE vs MAE gap (reveals outlier influence):")
print(f"  Model A: RMSE - MAE = {rmse_a - mae_a:.2f} kPa")
print(f"  Model B: RMSE - MAE = {rmse_b - mae_b:.2f} kPa")
print()
print("  A large RMSE-MAE gap means some predictions have very large errors.")
print("  If gap is small, errors are relatively uniform across samples.")

---

## Part 6: MAPE and Explained Variance

In [ ]:
# ─────────────────────────────────────────────────────
# MAPE AND EXPLAINED VARIANCE
# ─────────────────────────────────────────────────────

from sklearn.metrics import explained_variance_score, mean_absolute_percentage_error

mape_a = mean_absolute_percentage_error(y_test, y_pred_a) * 100
mape_b = mean_absolute_percentage_error(y_test, y_pred_b) * 100
ev_a = explained_variance_score(y_test, y_pred_a)
ev_b = explained_variance_score(y_test, y_pred_b)

print("MAPE (Mean Absolute Percentage Error):")
print(f"  Model A: {mape_a:.2f}%")
print(f"  Model B: {mape_b:.2f}%")
print(f"  Predictions are off by ~{min(mape_a, mape_b):.1f}% on average.")
print()
print("Explained Variance:")
print(f"  Model A: {ev_a:.4f}")
print(f"  Model B: {ev_b:.4f}")
print(f"  Close to R² when predictions are unbiased (mean residual near 0).")

---

## Part 7: Complete Metrics Summary Table

In [ ]:
# ─────────────────────────────────────────────────────
# COMPLETE METRICS COMPARISON TABLE
# ─────────────────────────────────────────────────────

metrics_df = pd.DataFrame({
    'Metric': ['R²', 'Adjusted R²', 'RMSE (kPa)', 'MAE (kPa)', 'MAPE (%)', 'Explained Var'],
    'Model A (RF)': [r2_a, adj_r2_a, rmse_a, mae_a, mape_a, ev_a],
    'Model B (XGB)': [r2_b, adj_r2_b, rmse_b, mae_b, mape_b, ev_b],
    'Better': ['Higher', 'Higher', 'Lower', 'Lower', 'Lower', 'Higher']
}).round(4)

# Determine winner for each metric
winners = []
for _, row in metrics_df.iterrows():
    if row['Better'] == 'Higher':
        winners.append('A' if row['Model A (RF)'] > row['Model B (XGB)'] else 'B')
    else:
        winners.append('A' if row['Model A (RF)'] < row['Model B (XGB)'] else 'B')
metrics_df['Winner'] = winners

print("\n" + "="*70)
print("  REGRESSION METRICS — Bearing Capacity Prediction")
print("="*70)
print(metrics_df.to_string(index=False))
print("="*70)

---

## Part 8: Residual Analysis

In [ ]:
# ─────────────────────────────────────────────────────
# RESIDUAL PLOTS — Diagnosing model behavior
# ─────────────────────────────────────────────────────

residuals_a = y_test.values - y_pred_a
residuals_b = y_test.values - y_pred_b

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Row 1: Residuals vs Predicted
for ax, res, pred, name, color in [
    (axes[0, 0], residuals_a, y_pred_a, 'Model A (RF)', '#3498db'),
    (axes[0, 1], residuals_b, y_pred_b, 'Model B (XGBoost)', '#e74c3c')
]:
    ax.scatter(pred, res, alpha=0.5, s=25, color=color, edgecolors='black', linewidth=0.3)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted qu (kPa)', fontsize=11)
    ax.set_ylabel('Residual (kPa)', fontsize=11)
    ax.set_title(f'{name} — Residuals vs Predicted', fontsize=12)

# Row 2: Residual distributions
for ax, res, name, color in [
    (axes[1, 0], residuals_a, 'Model A (RF)', '#3498db'),
    (axes[1, 1], residuals_b, 'Model B (XGBoost)', '#e74c3c')
]:
    ax.hist(res, bins=25, color=color, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
    ax.axvline(x=np.mean(res), color='red', linestyle='-', linewidth=1.5,
               label=f'Mean = {np.mean(res):.1f}')
    ax.set_xlabel('Residual (kPa)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{name} — Residual Distribution', fontsize=12)
    ax.legend(fontsize=10)

plt.suptitle('Residual Analysis: Are predictions biased?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Good signs: residuals centered at 0, no funnel shape, no patterns.")
print("Funnel shape = heteroscedasticity (error grows with prediction size).")

---

## Part 9: Actual vs Predicted Scatter Plot

In [ ]:
# ─────────────────────────────────────────────────────
# ACTUAL vs PREDICTED — The classic regression plot
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, name, rmse, r2, color in [
    (axes[0], y_pred_a, 'Model A (RF)', rmse_a, r2_a, '#3498db'),
    (axes[1], y_pred_b, 'Model B (XGBoost)', rmse_b, r2_b, '#e74c3c')
]:
    ax.scatter(y_test, y_pred, alpha=0.5, s=30, color=color,
               edgecolors='black', linewidth=0.3)

    # Perfect prediction line
    lims = [min(y_test.min(), y_pred.min()) - 50,
            max(y_test.max(), y_pred.max()) + 50]
    ax.plot(lims, lims, 'k--', linewidth=2, label='Perfect prediction')

    # +/- RMSE band
    ax.plot(lims, [l + rmse for l in lims], 'r:', linewidth=1, alpha=0.5)
    ax.plot(lims, [l - rmse for l in lims], 'r:', linewidth=1, alpha=0.5,
            label=f'+/- RMSE ({rmse:.0f} kPa)')

    ax.set_xlabel('Actual qu (kPa)', fontsize=12)
    ax.set_ylabel('Predicted qu (kPa)', fontsize=12)
    ax.set_title(f'{name}\nR²={r2:.4f} | RMSE={rmse:.1f} kPa', fontsize=12)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.legend(fontsize=10)
    ax.set_aspect('equal')

plt.suptitle('Actual vs Predicted Bearing Capacity',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Points close to the diagonal = good predictions.")
print("Points outside the +/- RMSE band = large individual errors.")

---

## Part 10: When Each Metric Matters — Decision Guide

In [ ]:
# ─────────────────────────────────────────────────────
# METRIC SELECTION GUIDE
# ─────────────────────────────────────────────────────

guide = pd.DataFrame({
    'Metric': ['R²', 'Adj R²', 'RMSE', 'MAE', 'MAPE'],
    'Use When': [
        'Comparing overall model fit',
        'Comparing models with different # of features',
        'Large errors are dangerous (structural safety)',
        'You want typical error, robust to outliers',
        'Reporting to non-technical stakeholders'
    ],
    'Geotech Example': [
        'Which model best predicts settlement?',
        'Does adding 10 more lab tests improve prediction?',
        'Bearing capacity: underestimate = foundation failure',
        'CPT-based soil classification: a few mismatches OK',
        'Client report: model is accurate within 8%'
    ],
    'Sensitive To': [
        'Scale of target, number of features',
        'Less sensitive to number of features',
        'Outlier errors (squares them)',
        'Robust to outlier errors',
        'Small actual values (divides by them)'
    ]
})

print("\nWhen to Use Each Regression Metric:")
print("="*90)
for _, row in guide.iterrows():
    print(f"\n{row['Metric']:>8}:")
    print(f"          Use:       {row['Use When']}")
    print(f"          Example:   {row['Geotech Example']}")
    print(f"          Caution:   {row['Sensitive To']}")
print("\n" + "="*90)

---

## Practice Exercises

### Exercise 1: Metric Sensitivity
Add 5 extreme outlier predictions (off by 500+ kPa) to `y_pred_a`. Recompute RMSE and MAE. Which metric changes more? Why?

### Exercise 2: Custom Metric
Create a custom metric that penalizes **under-predictions** twice as much as over-predictions (asymmetric loss). This is relevant for bearing capacity — underestimating is more dangerous.

### Exercise 3: MAPE Trap
What happens to MAPE when actual values are close to zero? Demonstrate with a small example.

In [ ]:
# EXERCISE 1: Outlier sensitivity
# y_pred_a_noisy = y_pred_a.copy()
# y_pred_a_noisy[:5] += 500  # Add 5 large errors
# rmse_noisy = np.sqrt(mean_squared_error(y_test, y_pred_a_noisy))
# mae_noisy = mean_absolute_error(y_test, y_pred_a_noisy)
# print(f"Original: RMSE={rmse_a:.1f}, MAE={mae_a:.1f}")
# print(f"Noisy:    RMSE={rmse_noisy:.1f}, MAE={mae_noisy:.1f}")

In [ ]:
# EXERCISE 2: Asymmetric loss
# def asymmetric_error(y_true, y_pred, penalty_under=2.0):
#     errors = y_true - y_pred
#     weights = np.where(errors > 0, penalty_under, 1.0)  # under-prediction gets 2x
#     return np.mean(weights * np.abs(errors))
#
# print(f"Asymmetric loss A: {asymmetric_error(y_test, y_pred_a):.2f}")
# print(f"Asymmetric loss B: {asymmetric_error(y_test, y_pred_b):.2f}")

---

## Key Takeaways

1. **R²** measures overall fit (0 to 1) but can be misleading with many features. Always report **Adjusted R²** when comparing models with different feature counts.

2. **RMSE** penalizes large errors heavily (squares them). Use it when large prediction errors are dangerous — such as underestimating bearing capacity.

3. **MAE** gives the typical prediction error and is robust to outliers. The gap between RMSE and MAE reveals whether your model has a few very large errors.

4. **MAPE** is intuitive for stakeholders ("off by 8%") but fails when actual values are near zero — the percentage blows up.

5. **Residual plots** reveal problems no single metric can: bias, heteroscedasticity, and patterns mean the model is missing something systematic.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| R² | Default sklearn regression scoring metric |
| RMSE | Most common loss for boosting (XGBoost, LightGBM) |
| MAE | Used as loss when robustness to outliers is needed |
| Residual analysis | Model diagnostics — checking assumptions |
| Actual vs Predicted | Standard regression evaluation plot |
| Metric selection | Choosing the right `scoring` in GridSearchCV |
| Asymmetric loss | Custom loss functions for domain-specific needs |

---

*Day 050 of 100 — One number never tells the whole story!*